<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/SDXL-LORA-TRAINING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDXL LoRA training (free Colab T4)

Trains an SDXL LoRA with [kohya-ss sd-scripts](https://github.com/kohya-ss/sd-scripts). The finished LoRA is saved to
`MyDrive/ComfyUI/loras_sdxl`, so it shows up in the website's LoRA list the next time you start the BigLust notebook.

**With the website (easiest)**
1. **Runtime → Change runtime type → T4 GPU.** Don't run the BigLust or Qwen notebook at the same time — training needs the whole GPU.
2. Run **cell 1** (setup, ~3 minutes; Colab asks for Google Drive access) and **cell 2** (prints a `READY:` URL).
3. On the website, open the **LoRA Training** tab, paste the URL, name your LoRA, pick a trigger word, drop in
   **15–40 images**, check the tags (Auto-tag helps), and press **Start training**. Every option has an ⓘ explaining it.
4. Training takes roughly 1–3 hours; the website shows the progress. Keep this Colab tab open.

**Without the website:** run cell 1, then fill in and run **cell 3** (it opens a file picker if your images aren't in Drive yet).

Your images and captions are stored in Google Drive under `MyDrive/ComfyUI/training/<name>/images`. Progress is saved
every 250 steps: if Colab disconnects, run cells 1–2 again and press Start training with the same name — it continues
from the last saved step. **Cell 4** shows the progress here in Colab too.

Only train on images you have the rights to use, and only train a LoRA of a real person with that person's consent.

In [ ]:
# Cell 1: setup — Google Drive, kohya-ss sd-scripts, the auto-tagger, and the shared training code
%cd /content
import os, re, io, json, glob, time, shutil, hashlib, threading, subprocess, collections

from google.colab import drive
drive.mount("/content/drive")  # Colab asks you to allow access

DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI"
TRAIN_ROOT = f"{DRIVE_ROOT}/training"      # <name>/images (+ captions) and <name>/output (checkpoints)
LORAS_SDXL = f"{DRIVE_ROOT}/loras_sdxl"    # finished LoRAs go here (the BigLust notebook reads this folder)
os.makedirs(TRAIN_ROOT, exist_ok=True)
os.makedirs(LORAS_SDXL, exist_ok=True)

# kohya-ss sd-scripts, pinned to a tested commit so updates can't break the notebook
SD_SCRIPTS = "/content/sd-scripts"
SD_SCRIPTS_COMMIT = "cb70b96"
if not os.path.exists(SD_SCRIPTS):
    !git clone -q https://github.com/kohya-ss/sd-scripts {SD_SCRIPTS}
!cd {SD_SCRIPTS} && git fetch -q origin && git checkout -q {SD_SCRIPTS_COMMIT}
!cd {SD_SCRIPTS} && pip install -q -r requirements.txt
!pip install -q onnx onnxruntime-gpu fastapi uvicorn nest_asyncio
!apt install -qqy aria2 > /dev/null

os.makedirs("/content/base_models", exist_ok=True)
VAE_PATH = "/content/base_models/sdxl_vae.safetensors"
if not os.path.exists(VAE_PATH):
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/stabilityai/sdxl-vae/resolve/main/sdxl_vae.safetensors -d /content/base_models -o sdxl_vae.safetensors
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

# ---------------------------------------------------------------- shared training code
CIVITAI_TOKEN = "777a5632707c15ecf54e13eb3b1b862b"
BIGLUST_URL = f"https://civitai.com/api/download/models/1081768?fileId=986850&token={CIVITAI_TOKEN}"
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
TAGGER_REPO = "SmilingWolf/wd-swinv2-tagger-v3"

DEFAULTS = {
    "trigger_word": "",
    "class_word": "woman",
    "add_trigger_to_captions": True,
    "base_model": "biglust",        # or a Civitai / Hugging Face link, or a file path
    "max_train_steps": 1500,
    "num_repeats": 10,
    "train_batch_size": 2,          # 1 if you get "CUDA out of memory"
    "resolution": 1024,
    "network_dim": 8,
    "network_alpha": 4,
    "learning_rate": 3e-4,
    "seed": 42,
    "save_every_n_steps": 250,
}


def clean_name(name):
    return re.sub(r"[^\w\-]+", "_", (name or "").strip()).strip("_")


def clean_file(file):
    file = os.path.basename(file or "")
    stem, ext = os.path.splitext(file)
    stem = re.sub(r"[^\w\-. ]+", "_", stem).strip(" .") or "image"
    return stem + ext.lower()


def images_dir(name):
    n = clean_name(name)
    if not n:
        raise ValueError("Give the LoRA a name first")
    return f"{TRAIN_ROOT}/{n}/images"


def output_dir(name):
    return f"{TRAIN_ROOT}/{clean_name(name)}/output"


def list_images(name):
    d = images_dir(name)
    return sorted(f for f in os.listdir(d) if f.lower().endswith(IMAGE_EXTS)) if os.path.isdir(d) else []


def caption_path(name, file):
    return os.path.join(images_dir(name), os.path.splitext(clean_file(file))[0] + ".txt")


def read_caption(name, file):
    p = caption_path(name, file)
    return open(p, encoding="utf-8").read().strip() if os.path.exists(p) else ""


def write_caption(name, file, caption):
    with open(caption_path(name, file), "w", encoding="utf-8") as fh:
        fh.write((caption or "").strip())


def split_tags(caption):
    return [t.strip() for t in (caption or "").split(",") if t.strip()]


def list_datasets():
    out = []
    for n in sorted(os.listdir(TRAIN_ROOT)):
        if os.path.isdir(os.path.join(TRAIN_ROOT, n, "images")):
            out.append({"name": n, "images": len(list_images(n)),
                        "has_lora": os.path.exists(f"{LORAS_SDXL}/{n}.safetensors")})
    return out


# ---- one background task at a time (training or auto-tagging), shared with the website
STATE = {"status": "idle", "task": None, "name": None, "phase": "", "step": 0, "total": 0,
         "elapsed": "", "eta": "", "error": None, "result": None, "started": None,
         "log": collections.deque(maxlen=60)}
_PROC = {"p": None, "stop": False}
PROGRESS_RE = re.compile(r"steps:\s*\d+%.*?\|\s*(\d+)/(\d+)\s*\[([^<\]]*)<([^,\]]*)")


def status_payload():
    s = {k: v for k, v in STATE.items() if k != "log"}
    s["log"] = list(STATE["log"])[-25:]
    return s


def _log(line):
    line = line.rstrip()
    if line:
        STATE["log"].append(line)


def _run(cmd, cwd):
    """Run a command, stream its output into STATE (training progress bars update step/eta)."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    _PROC["p"] = p
    buf = b""
    while True:
        chunk = p.stdout.read(4096)
        if not chunk:
            break
        buf += chunk
        parts = re.split(rb"[\r\n]", buf)
        buf = parts.pop()
        for raw in parts:
            line = raw.decode("utf-8", "replace").strip()
            if not line:
                continue
            m = PROGRESS_RE.search(line)
            if m:
                STATE.update(step=int(m.group(1)), total=int(m.group(2)),
                             elapsed=m.group(3).strip(), eta=m.group(4).strip(), phase="Training")
            else:
                _log(line)
    rc = p.wait()
    _PROC["p"] = None
    return rc


def busy():
    return STATE["status"] == "running"


def _begin(task, name, phase):
    if busy():
        raise RuntimeError(f"Busy: {STATE['task']} of '{STATE['name']}' is still running")
    STATE["log"].clear()
    STATE.update(status="running", task=task, name=clean_name(name), phase=phase, step=0, total=0,
                 elapsed="", eta="", error=None, result=None, started=time.time())
    _PROC["stop"] = False


def _finish(error=None, result=None):
    if _PROC["stop"]:
        STATE.update(status="stopped", phase="Stopped")
    elif error:
        STATE.update(status="error", error=str(error), phase="Failed")
    else:
        STATE.update(status="done", result=result, phase="Done")


def stop_task():
    _PROC["stop"] = True
    p = _PROC["p"]
    if p and p.poll() is None:
        p.terminate()


def ensure_base_model(base):
    base = (base or "biglust").strip()
    if base.lower() == "biglust":
        base = BIGLUST_URL
    if base.startswith("http"):
        url = base.replace("/blob/", "/resolve/")
        if "civitai.com" in url and "token=" not in url:
            url += ("&" if "?" in url else "?") + "token=" + CIVITAI_TOKEN
        path = f"/content/base_models/base_{hashlib.sha1(url.encode()).hexdigest()[:10]}.safetensors"
        if not os.path.exists(path):
            STATE["phase"] = "Downloading the base model (~6.5 GB, a few minutes)"
            if _run(["wget", "-q", "-O", path + ".part", url], "/content") != 0:
                raise RuntimeError("Could not download the base model — check the link")
            os.replace(path + ".part", path)
    else:
        path = base
    if not os.path.exists(path) or os.path.getsize(path) < 1_000_000_000:
        raise RuntimeError(f"Base model missing or incomplete: {path}")
    return path


def prepare_dataset(name, s):
    """Copy images + captions to the runtime (Drive files are not changed) and write the dataset config."""
    imgs = list_images(name)
    if not imgs:
        raise RuntimeError(f"No images for '{name}' yet — add 15-40 images first")
    local = f"/content/dataset/{clean_name(name)}"
    shutil.rmtree(local, ignore_errors=True)
    os.makedirs(local)
    trigger, cls = s["trigger_word"].strip(), s["class_word"].strip()
    default_caption = ", ".join(w for w in (trigger, cls) if w)
    for f in imgs:
        shutil.copy2(os.path.join(images_dir(name), f), os.path.join(local, f))
        caption = read_caption(name, f)
        if not caption:
            caption = default_caption
        elif s["add_trigger_to_captions"] and trigger and trigger.lower() not in caption.lower():
            caption = f"{trigger}, {caption}"
        with open(os.path.join(local, os.path.splitext(f)[0] + ".txt"), "w", encoding="utf-8") as fh:
            fh.write(caption)
    toml_path = "/content/dataset/dataset.toml"
    with open(toml_path, "w") as fh:
        fh.write(f'''[general]
enable_bucket = true
min_bucket_reso = 512
max_bucket_reso = {max(s["resolution"] * 3 // 2, 1024)}
bucket_reso_steps = 64

[[datasets]]
resolution = {s["resolution"]}
batch_size = {s["train_batch_size"]}

  [[datasets.subsets]]
  image_dir = "{local}"
  num_repeats = {s["num_repeats"]}
  caption_extension = ".txt"
''')
    return toml_path, len(imgs)


def build_train_args(name, s, base_path, toml_path):
    n, out = clean_name(name), output_dir(name)
    os.makedirs(out, exist_ok=True)
    args = [
        "python", "sdxl_train_network.py",
        f"--pretrained_model_name_or_path={base_path}", f"--vae={VAE_PATH}",
        f"--dataset_config={toml_path}", f"--output_dir={out}", f"--output_name={n}",
        "--save_model_as=safetensors", "--save_precision=fp16",
        "--network_module=networks.lora", f"--network_dim={s['network_dim']}", f"--network_alpha={s['network_alpha']}",
        # Fits a free T4: fp16 straight to the GPU, U-Net only with cached text-encoder outputs + latents,
        # gradient checkpointing, 8-bit AdamW, SDPA attention, fp32 VAE (fp16 SDXL VAE gives NaNs)
        "--network_train_unet_only", "--cache_text_encoder_outputs", "--cache_latents", "--cache_latents_to_disk",
        f"--learning_rate={s['learning_rate']}", f"--unet_lr={s['learning_rate']}",
        "--lr_scheduler=cosine", f"--lr_warmup_steps={max(1, s['max_train_steps'] // 20)}",
        "--optimizer_type=AdamW8bit", f"--max_train_steps={s['max_train_steps']}",
        "--mixed_precision=fp16", "--full_fp16", "--lowram", "--no_half_vae",
        "--gradient_checkpointing", "--sdpa", "--max_data_loader_n_workers=1", f"--seed={s['seed']}",
        f"--save_every_n_steps={s['save_every_n_steps']}", "--save_state",
        f"--save_last_n_steps_state={s['save_every_n_steps'] * 2}",
    ]
    states = sorted(glob.glob(f"{out}/{n}-step*-state"), key=lambda p: int(re.search(r"-step(\d+)-state", p).group(1)))
    if states:  # continue an interrupted run with the same name
        args.append(f"--resume={states[-1]}")
        _log(f"Resuming from {os.path.basename(states[-1])}")
    return args


def merge_settings(settings):
    s = dict(DEFAULTS)
    for k, v in (settings or {}).items():
        if k in DEFAULTS and v is not None and v != "":
            t = type(DEFAULTS[k])
            if t is bool:
                s[k] = v if isinstance(v, bool) else str(v).lower() in ("1", "true", "yes")
            elif t is int:
                s[k] = int(float(v))
            else:
                s[k] = t(v)
    s["train_batch_size"] = max(1, min(s["train_batch_size"], 4))
    s["max_train_steps"] = max(50, s["max_train_steps"])
    s["save_every_n_steps"] = max(50, s["save_every_n_steps"])
    return s


def _train_worker(name, s):
    try:
        base_path = ensure_base_model(s["base_model"])
        STATE["phase"] = "Preparing images"
        toml_path, n = prepare_dataset(name, s)
        _log(f"{n} images, {s['max_train_steps']} steps, batch {s['train_batch_size']}, {s['resolution']} px")
        STATE.update(phase="Loading model and caching images (a few minutes)", total=s["max_train_steps"])
        rc = _run(build_train_args(name, s, base_path, toml_path), SD_SCRIPTS)
        final = f"{output_dir(name)}/{clean_name(name)}.safetensors"
        if rc == 0 and os.path.exists(final):
            dest = f"{LORAS_SDXL}/{clean_name(name)}.safetensors"
            shutil.copy2(final, dest)
            _finish(result={"lora": os.path.basename(dest), "path": dest})
        else:
            tail = " | ".join(list(STATE["log"])[-3:])
            _finish(error=f"Training failed (exit code {rc}). Last output: {tail}")
    except Exception as e:
        _finish(error=e)


def start_training(name, settings=None):
    s = merge_settings(settings)
    _begin("training", name, "Starting")
    STATE["settings"] = s
    threading.Thread(target=_train_worker, args=(name, s), daemon=True).start()
    return s


def _autotag_worker(name, threshold, mode, trigger):
    try:
        imgs = list_images(name)
        if not imgs:
            raise RuntimeError("No images to tag")
        work = f"/content/autotag/{clean_name(name)}"
        shutil.rmtree(work, ignore_errors=True)
        os.makedirs(work)
        for f in imgs:
            shutil.copy2(os.path.join(images_dir(name), f), os.path.join(work, f))
        STATE["phase"] = "Auto-tagging (the first run downloads the tagger, ~0.5 GB)"
        rc = _run(["python", "finetune/tag_images_by_wd14_tagger.py", work, "--onnx", "--repo_id", TAGGER_REPO,
                   "--model_dir", "/content/wd14_models", "--batch_size", "4", "--caption_extension", ".txt",
                   "--thresh", str(threshold), "--remove_underscore"], SD_SCRIPTS)
        if rc != 0:
            raise RuntimeError("The tagger failed — see the log")
        changed = 0
        for f in imgs:
            p = os.path.join(work, os.path.splitext(f)[0] + ".txt")
            if not os.path.exists(p):
                continue
            new_tags = split_tags(open(p, encoding="utf-8").read())
            tags = split_tags(read_caption(name, f)) if mode == "append" else []
            for t in new_tags:
                if t.lower() not in {x.lower() for x in tags}:
                    tags.append(t)
            if trigger:
                tags = [trigger] + [t for t in tags if t.lower() != trigger.lower()]
            write_caption(name, f, ", ".join(tags))
            changed += 1
        _finish(result={"tagged": changed})
    except Exception as e:
        _finish(error=e)


def start_autotag(name, threshold=0.35, mode="append", trigger=""):
    _begin("auto-tagging", name, "Starting the tagger")
    threading.Thread(target=_autotag_worker, args=(name, float(threshold), mode, (trigger or "").strip()),
                     daemon=True).start()


print("Setup done. Run cell 2 to use the website, or cell 3 to train without it.")

In [ ]:
# Cell 2: training server for the website (LoRA Training tab). Paste the READY: URL there.
# This cell finishes after printing the URL; the server keeps running in the background.
if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import base64, io, re, subprocess, threading, time
import nest_asyncio, uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
from PIL import Image, ImageOps

nest_asyncio.apply()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


def fail(msg, code=400):
    raise HTTPException(status_code=code, detail=str(msg))


def dataset_payload(name):
    return {"name": clean_name(name),
            "images": [{"file": f, "caption": read_caption(name, f)} for f in list_images(name)]}


@app.get("/")
def root():
    return {"status": "ok", "mode": "lora_training"}


@app.get("/training/info")
def info():
    return {"mode": "lora_training", "defaults": DEFAULTS, "datasets": list_datasets(), "status": status_payload()}


@app.get("/training/status")
def status():
    return status_payload()


@app.get("/training/dataset/{name}")
def get_dataset(name: str):
    try:
        return dataset_payload(name)
    except ValueError as e:
        fail(e)


class Upload(BaseModel):
    filename: str
    data: str  # base64 (data: URL prefix allowed)
    caption: str | None = None


@app.post("/training/dataset/{name}/upload")
def upload(name: str, req: Upload):
    try:
        d = images_dir(name)
    except ValueError as e:
        fail(e)
    file = clean_file(req.filename)
    if not file.lower().endswith(IMAGE_EXTS):
        fail(f"{req.filename}: only jpg, png, webp and bmp images are supported")
    raw = base64.b64decode(re.sub(r"^data:[^,]*,", "", req.data))
    try:
        Image.open(io.BytesIO(raw)).verify()
    except Exception:
        fail(f"{req.filename} is not a readable image")
    os.makedirs(d, exist_ok=True)
    stem, ext = os.path.splitext(file)
    i = 1
    while os.path.exists(os.path.join(d, file)):  # never overwrite: photo.jpg -> photo-1.jpg
        file, i = f"{stem}-{i}{ext}", i + 1
    with open(os.path.join(d, file), "wb") as fh:
        fh.write(raw)
    if req.caption is not None:
        write_caption(name, file, req.caption)
    return {"file": file, "caption": read_caption(name, file)}


class Caption(BaseModel):
    file: str
    caption: str


@app.put("/training/dataset/{name}/caption")
def set_caption(name: str, req: Caption):
    if clean_file(req.file) not in list_images(name):
        fail("Image not found", 404)
    write_caption(name, req.file, req.caption)
    return {"file": req.file, "caption": read_caption(name, req.file)}


class Bulk(BaseModel):
    action: str  # "add" (to the start) or "remove"
    tag: str


@app.post("/training/dataset/{name}/bulk")
def bulk(name: str, req: Bulk):
    tag = req.tag.strip().strip(",").strip()
    if not tag:
        fail("Type a tag first")
    for f in list_images(name):
        tags = [t for t in split_tags(read_caption(name, f)) if t.lower() != tag.lower()]
        if req.action == "add":
            tags.insert(0, tag)
        write_caption(name, f, ", ".join(tags))
    return dataset_payload(name)


@app.delete("/training/dataset/{name}/image/{file}")
def delete_image(name: str, file: str):
    file = clean_file(file)
    if file not in list_images(name):
        fail("Image not found", 404)
    os.remove(os.path.join(images_dir(name), file))
    if os.path.exists(caption_path(name, file)):
        os.remove(caption_path(name, file))
    return dataset_payload(name)


THUMBS = "/content/thumbs"


@app.get("/training/dataset/{name}/thumb/{file}")
def thumb(name: str, file: str):
    file = clean_file(file)
    src = os.path.join(images_dir(name), file)
    if not os.path.exists(src):
        fail("Image not found", 404)
    key = hashlib.sha1(f"{src}|{os.path.getmtime(src)}".encode()).hexdigest()
    path = os.path.join(THUMBS, key + ".jpg")
    if not os.path.exists(path):
        os.makedirs(THUMBS, exist_ok=True)
        im = ImageOps.exif_transpose(Image.open(src)).convert("RGB")
        im.thumbnail((360, 360))
        im.save(path, "JPEG", quality=85)
    return Response(open(path, "rb").read(), media_type="image/jpeg", headers={"Cache-Control": "max-age=3600"})


class AutoTag(BaseModel):
    threshold: float = 0.35
    mode: str = "append"  # "append" to existing captions or "replace" them
    trigger_word: str = ""


@app.post("/training/dataset/{name}/autotag")
def autotag(name: str, req: AutoTag):
    try:
        start_autotag(name, req.threshold, req.mode, req.trigger_word)
    except Exception as e:
        fail(e)
    return status_payload()


class Start(BaseModel):
    name: str
    settings: dict = {}


@app.post("/training/start")
def start(req: Start):
    try:
        start_training(req.name, req.settings)
    except Exception as e:
        fail(e)
    return status_payload()


@app.post("/training/stop")
def stop():
    stop_task()
    return status_payload()


if globals().get("TRAINING_SERVER_URL"):
    print(f"The server is already running: {TRAINING_SERVER_URL}")
else:
    threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
                     daemon=True).start()
    time.sleep(4)
    cf = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url, t0 = None, time.time()
    while time.time() - t0 < 60:
        line = cf.stdout.readline()
        if not line:
            break
        m = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    threading.Thread(target=lambda: [None for _ in cf.stdout], daemon=True).start()  # keep the pipe drained
    if url:
        TRAINING_SERVER_URL = url
        print("=" * 60)
        print(f"READY: {url}")
        print("Paste this into the website: LoRA Training tab → Training server URL.")
        print("Keep this Colab tab open while you work. Progress is also printed by cell 4 if you want it here.")
        print("=" * 60)
    else:
        print("Could not start the Cloudflare tunnel — run this cell again.")

In [ ]:
# Cell 3 (optional): train WITHOUT the website — fill in the form and run. Skip this if you use the website.
#@markdown Images: `MyDrive/ComfyUI/training/<name>/images` (this cell opens a file picker if the folder is empty).
LORA_NAME = "my_lora"  #@param {type:"string"}
TRIGGER_WORD = "zxq_subject"  #@param {type:"string"}
CLASS_WORD = "woman"  #@param {type:"string"}
BASE_MODEL = "biglust"  #@param {type:"string"}
MAX_TRAIN_STEPS = 1500  #@param {type:"integer"}
TRAIN_BATCH_SIZE = 2  #@param {type:"integer"}
NETWORK_DIM = 8  #@param {type:"integer"}
NETWORK_ALPHA = 4  #@param {type:"integer"}
LEARNING_RATE = 3e-4  #@param {type:"number"}

if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import os, io, zipfile, time
from google.colab import files

name = clean_name(LORA_NAME)
d = images_dir(name)
os.makedirs(d, exist_ok=True)
if not list_images(name):
    print(f"Choose images, caption .txt files or one .zip. They'll be saved to {d}")
    for fname, data in files.upload().items():
        if fname.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for m in z.namelist():
                    base = os.path.basename(m)
                    if base and not base.startswith(".") and "__MACOSX" not in m and base.lower().endswith(IMAGE_EXTS + (".txt",)):
                        open(os.path.join(d, clean_file(base)), "wb").write(z.read(m))
        elif fname.lower().endswith(IMAGE_EXTS + (".txt",)):
            open(os.path.join(d, clean_file(fname)), "wb").write(data)
        if os.path.exists(fname):
            os.remove(fname)
print(f"{len(list_images(name))} images in {d}")

start_training(name, {"trigger_word": TRIGGER_WORD, "class_word": CLASS_WORD, "base_model": BASE_MODEL,
                      "max_train_steps": MAX_TRAIN_STEPS, "train_batch_size": TRAIN_BATCH_SIZE,
                      "network_dim": NETWORK_DIM, "network_alpha": NETWORK_ALPHA, "learning_rate": LEARNING_RATE})
last = None
while STATE["status"] == "running":
    msg = f"{STATE['phase']}" + (f" — step {STATE['step']}/{STATE['total']}, {STATE['eta']} left" if STATE["step"] else "")
    if msg != last:
        print(time.strftime("%H:%M"), msg)
        last = msg
    time.sleep(20)
print(STATE["status"], STATE.get("result") or STATE.get("error") or "")
if STATE["status"] == "done":
    print(f"Start the BigLust notebook and pick {STATE['result']['lora']} in the website's LoRA list; use '{TRIGGER_WORD}' in prompts.")

In [ ]:
# Cell 4 (optional): show the current task's progress and log here in Colab
if "clean_name" not in globals():
    raise RuntimeError("Run cell 1 (setup) first, then run this cell again.")

import time
from IPython.display import clear_output
try:
    while True:
        s = status_payload()
        clear_output(wait=True)
        print(f"{s['task'] or 'nothing running'} {s['name'] or ''}: {s['status']} — {s['phase']}")
        if s["total"]:
            print(f"step {s['step']}/{s['total']}  elapsed {s['elapsed']}  left {s['eta']}")
        print("\n".join(s["log"][-15:]))
        if s["status"] != "running":
            break
        time.sleep(10)
except KeyboardInterrupt:
    pass